In [1]:
import rasterio
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
print("="*60)
print("DATA EXTRACTION FOR CALIFORNIA")
print("="*60)

DATA EXTRACTION FOR CALIFORNIA


1. DEFINE REGIONS AND THEIR FILES

In [3]:
# REGION 1: SOUTH CALIFORNIA (Fresno + Kings County border)
# Your Sentinel tiles for South California
SOUTH_SENTINEL_TILES = [
    r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_South-1.tif",
    r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_South-2.tif",
    r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_South-3.tif",
    r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_South-4.tif",
]

# CDL files for South California (Fresno and Kings)
SOUTH_CDL_FRESNO = r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\cdl\californiaFresno.tif"
SOUTH_CDL_KINGS = r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\cdl\californiaKings.tif"

# REGION 2: NORTH CALIFORNIA (Yolo County)
NORTH_SENTINEL_TILES = [
    r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_North-1.tif",
    r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_North-2.tif",
    r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_North-3.tif",
    r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_North-4.tif",
]

# CDL for North California (Yolo County)
NORTH_CDL = r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\cdl\californiaNorth.tif"

# Output directories
OUTPUT_DIR_SOUTH = r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\processed\california\south"
OUTPUT_DIR_NORTH = r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\processed\california\north"
OUTPUT_DIR_COMBINED = r"C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\processed\california"

os.makedirs(OUTPUT_DIR_SOUTH, exist_ok=True)
os.makedirs(OUTPUT_DIR_NORTH, exist_ok=True)
os.makedirs(OUTPUT_DIR_COMBINED, exist_ok=True)

print("\n✅ Paths configured")



✅ Paths configured


2. HELPER FUNCTIONS

In [4]:
def merge_cdls(cdl_paths, sentinel_bounds, sentinel_crs, sentinel_transform, sentinel_shape):
    """
    Merge multiple CDL files (Fresno + Kings) for a given Sentinel tile.
    Returns resampled CDL matching Sentinel shape.
    """
    from rasterio.warp import transform_bounds, reproject, Resampling
    from rasterio.windows import from_bounds
    
    merged_cdl = None
    final_transform = None
    final_crs = None
    
    for cdl_path in cdl_paths:
        with rasterio.open(cdl_path) as cdl_src:
            # Transform Sentinel bounds to CDL CRS
            bounds_in_cdl_crs = transform_bounds(
                sentinel_crs, cdl_src.crs,
                sentinel_bounds.left, sentinel_bounds.bottom,
                sentinel_bounds.right, sentinel_bounds.top
            )
            
            # Create window
            window = from_bounds(
                bounds_in_cdl_crs[0], bounds_in_cdl_crs[1],
                bounds_in_cdl_crs[2], bounds_in_cdl_crs[3],
                cdl_src.transform
            )
            
            # Read cropped CDL
            cdl_cropped = cdl_src.read(1, window=window)
            cdl_transform = cdl_src.window_transform(window)
            
            # Skip if this CDL has no data for this tile
            if cdl_cropped.size == 0:
                print(f"    ⚠️ CDL {os.path.basename(cdl_path)} has no overlap - skipping")
                continue
            
            if merged_cdl is None:
                merged_cdl = cdl_cropped
                final_transform = cdl_transform
                final_crs = cdl_src.crs
            else:
                # Only combine if shapes match
                if cdl_cropped.shape == merged_cdl.shape:
                    mask = cdl_cropped > 0
                    merged_cdl = np.where(mask, cdl_cropped, merged_cdl)
                else:
                    print(f"    ⚠️ Shape mismatch - using first CDL only")
    
    if merged_cdl is None:
        print(f"  ⚠️ No CDL data for this tile!")
        return None, None
    
    # Resample to Sentinel resolution
    cdl_resampled = np.zeros(sentinel_shape, dtype=np.uint8)
    
    reproject(
        source=merged_cdl,
        destination=cdl_resampled,
        src_transform=final_transform,
        src_crs=final_crs,
        dst_transform=sentinel_transform,
        dst_crs=sentinel_crs,
        resampling=Resampling.nearest
    )
    
    return cdl_resampled, sentinel_transform

def process_region(sentinel_tiles, cdl_paths, region_name, output_dir, crops_to_keep):
    """
    Process a region (South or North) and extract samples.
    """
    print(f"\n{'='*60}")
    print(f"PROCESSING {region_name.upper()}")
    print(f"{'='*60}")
    
    all_X = []
    all_y = []
    all_masks = []
    
    for tile_idx, tile_path in enumerate(sentinel_tiles, 1):
        print(f"\n--- Processing Tile {tile_idx}/{len(sentinel_tiles)} ---")
        print(f"File: {tile_path}")
        
        # Load Sentinel-2 tile
        with rasterio.open(tile_path) as src:
            sentinel = src.read()  # Shape: (bands, height, width)
            sentinel_bounds = src.bounds
            sentinel_transform = src.transform
            sentinel_crs = src.crs
            sentinel_shape = sentinel.shape[1:]
            print(f"  Sentinel shape: {sentinel.shape}")
            print(f"  Sentinel CRS: {sentinel_crs}")
            print(f"  Sentinel bounds: {sentinel_bounds}")
        
        # Load and merge CDL(s) for this tile
        if isinstance(cdl_paths, list):
            # South: merge Fresno + Kings
            cdl_resampled, _ = merge_cdls(cdl_paths, sentinel_bounds, sentinel_crs, sentinel_transform, sentinel_shape)
        else:
            # North: single CDL
            with rasterio.open(cdl_paths) as cdl_src:
                from rasterio.warp import transform_bounds, reproject, Resampling
                from rasterio.windows import from_bounds
                
                bounds_in_cdl_crs = transform_bounds(
                    sentinel_crs, cdl_src.crs,
                    sentinel_bounds.left, sentinel_bounds.bottom,
                    sentinel_bounds.right, sentinel_bounds.top
                )
                
                window = from_bounds(
                    bounds_in_cdl_crs[0], bounds_in_cdl_crs[1],
                    bounds_in_cdl_crs[2], bounds_in_cdl_crs[3],
                    cdl_src.transform
                )
                
                cdl_cropped = cdl_src.read(1, window=window)
                cdl_transform = cdl_src.window_transform(window)
                
                cdl_resampled = np.zeros(sentinel_shape, dtype=np.uint8)
                
                reproject(
                    source=cdl_cropped,
                    destination=cdl_resampled,
                    src_transform=cdl_transform,
                    src_crs=cdl_src.crs,
                    dst_transform=sentinel_transform,
                    dst_crs=sentinel_crs,
                    resampling=Resampling.nearest
                )
        
        if cdl_resampled is None:
            print(f"  ⚠️ No CDL data for this tile! Skipping...")
            continue
        
        print(f"  Resampled CDL shape: {cdl_resampled.shape}")
        
        # Filter to keep only desired crops
        valid_mask = np.isin(cdl_resampled, crops_to_keep)
        valid_rows, valid_cols = np.where(valid_mask)
        print(f"  Valid cropland pixels with target crops: {len(valid_rows):,}")
        
        if len(valid_rows) == 0:
            print(f"  ⚠️ No valid cropland pixels! Skipping...")
            continue
        
        # Extract samples
        X_tile = []
        y_tile = []
        masks_tile = []
        
        n_samples = min(1250, len(valid_rows))
        sample_indices = np.random.choice(len(valid_rows), n_samples, replace=False)
        
        print(f"  Extracting {n_samples} samples...")
        
        for idx in tqdm(sample_indices, desc=f"  Tile {tile_idx}"):
            row = valid_rows[idx]
            col = valid_cols[idx]
            
            # Extract time series for this pixel
            pixel_ts = sentinel[:, row, col]
            
            # Normalize: divide by 10000 (as per paper)
            pixel_ts = pixel_ts / 10000.0
            
            # Create mask: 1 = data exists, 0 = missing
            mask = (pixel_ts != 0).astype(np.float32)
            
            # Use first 360 bands (36×10)
            if len(pixel_ts) >= 360:
                pixel_ts = pixel_ts[:360].reshape(36, 10)
                mask = mask[:360].reshape(36, 10)
            else:
                continue
            
            X_tile.append(pixel_ts)
            masks_tile.append(mask)
            y_tile.append(cdl_resampled[row, col])
        
        if len(X_tile) == 0:
            continue
        
        all_X.append(np.array(X_tile))
        all_masks.append(np.array(masks_tile))
        all_y.append(np.array(y_tile))
        
        print(f"  Tile {tile_idx} extracted: {len(X_tile)} samples")
    
    if len(all_X) == 0:
        print(f"  ⚠️ No samples extracted for {region_name}!")
        return None, None, None
    
    # Combine all tiles for this region
    X = np.concatenate(all_X, axis=0)
    masks = np.concatenate(all_masks, axis=0)
    y = np.concatenate(all_y, axis=0)
    
    # Save region-specific data
    np.save(os.path.join(output_dir, 'X.npy'), X)
    np.save(os.path.join(output_dir, 'masks.npy'), masks)
    np.save(os.path.join(output_dir, 'y.npy'), y)
    
    print(f"\n✅ {region_name.upper()} saved to: {output_dir}")
    print(f"   X shape: {X.shape}")
    print(f"   y shape: {y.shape}")
    
    return X, masks, y

# 3. DEFINE CROPS TO KEEP PER REGION

In [5]:
# From paper Table 2 for California
# South California crops (San Joaquin Valley)
SOUTH_CROPS = [36, 57, 69, 74, 204]  # Alfalfa, Rice, Grapes, Pistachios, Almonds

# North California crops (Sacramento Valley)
NORTH_CROPS = [57, 204]  # Rice, Almonds (add others as needed)

print("\n" + "="*60)
print("CROP FILTERS")
print("="*60)
print(f"South California crops: {SOUTH_CROPS}")
print(f"North California crops: {NORTH_CROPS}")



CROP FILTERS
South California crops: [36, 57, 69, 74, 204]
North California crops: [57, 204]


4. PROCESS SOUTH CALIFORNIA

In [6]:
# South uses TWO CDL files (Fresno + Kings)
SOUTH_CDL_PATHS = [SOUTH_CDL_FRESNO, SOUTH_CDL_KINGS]

X_south, masks_south, y_south = process_region(
    sentinel_tiles=SOUTH_SENTINEL_TILES,
    cdl_paths=SOUTH_CDL_PATHS,
    region_name="South California",
    output_dir=OUTPUT_DIR_SOUTH,
    crops_to_keep=SOUTH_CROPS
)



PROCESSING SOUTH CALIFORNIA

--- Processing Tile 1/4 ---
File: C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_South-1.tif
  Sentinel shape: (370, 1792, 1792)
  Sentinel CRS: EPSG:4326
  Sentinel bounds: BoundingBox(left=-119.91134620610909, bottom=36.629075204558724, right=-119.75036810719487, top=36.790053303472945)
    ⚠️ CDL californiaKings.tif has no overlap - skipping
  Resampled CDL shape: (1792, 1792)
  Valid cropland pixels with target crops: 603,521
  Extracting 1250 samples...


  Tile 1: 100%|██████████| 1250/1250 [00:00<00:00, 19132.36it/s]

  Tile 1 extracted: 1250 samples

--- Processing Tile 2/4 ---
File: C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_South-2.tif


  Sentinel shape: (370, 1792, 691)
  Sentinel CRS: EPSG:4326
  Sentinel bounds: BoundingBox(left=-119.75036810719487, bottom=36.629075204558724, right=-119.68829452106222, top=36.790053303472945)
    ⚠️ CDL californiaKings.tif has no overlap - skipping
  Resampled CDL shape: (1792, 691)
  Valid cropland pixels with target crops: 202,186
  Extracting 1250 samples...


  Tile 2: 100%|██████████| 1250/1250 [00:00<00:00, 27570.02it/s]

...

  Tile 3: 100%|██████████| 1250/1250 [00:00<00:00, 39093.59it/s]

  Tile 3 extracted: 1250 samples

--- Processing Tile 4/4 ---
File: C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_South-4.tif


  Sentinel shape: (370, 212, 691)
  Sentinel CRS: EPSG:4326
  Sentinel bounds: BoundingBox(left=-119.75036810719487, bottom=36.61003092053539, right=-119.68829452106222, top=36.629075204558724)
    ⚠️ CDL californiaKings.tif has no overlap - skipping
  Resampled CDL shape: (212, 691)
  Valid cropland pixels with target crops: 98,402
  Extracting 1250 samples...


  Tile 4: 100%|██████████| 1250/1250 [00:00<00:00, 41789.92it/s]

  Tile 4 extracted: 1250 samples

✅ SOUTH CALIFORNIA saved to: C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\processed\california\south
   X shape: (5000, 36, 10)
   y shape: (5000,)


### 5. PROCESS NORTH CALIFORNIA

In [7]:
X_north, masks_north, y_north = process_region(
    sentinel_tiles=NORTH_SENTINEL_TILES,
    cdl_paths=NORTH_CDL,
    region_name="North California",
    output_dir=OUTPUT_DIR_NORTH,
    crops_to_keep=NORTH_CROPS
)


PROCESSING NORTH CALIFORNIA

--- Processing Tile 1/4 ---
File: C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_North-1.tif
  Sentinel shape: (370, 1792, 1792)
  Sentinel CRS: EPSG:4326
  Sentinel bounds: BoundingBox(left=-121.81406780940264, bottom=38.42902953934901, right=-121.65308971048843, top=38.59000763826323)
  Resampled CDL shape: (1792, 1792)
  Valid cropland pixels with target crops: 17,315
  Extracting 1250 samples...


  Tile 1: 100%|██████████| 1250/1250 [00:00<00:00, 26473.04it/s]

  Tile 1 extracted: 1250 samples

--- Processing Tile 2/4 ---
File: C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_North-2.tif


  Sentinel shape: (370, 1792, 752)
  Sentinel CRS: EPSG:4326
  Sentinel bounds: BoundingBox(left=-121.65308971048843, bottom=38.42902953934901, right=-121.58553640112264, top=38.59000763826323)
  Resampled CDL shape: (1792, 752)
  Valid cropland pixels with target crops: 244
  Extracting 244 samples...


  Tile 2: 100%|██████████| 244/244 [00:00<00:00, 55084.24it/s]

...

  Tile 3: 100%|██████████| 11/11 [00:00<00:00, 33168.47it/s]

  Tile 3 extracted: 11 samples

--- Processing Tile 4/4 ---
File: C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_400km2_California_North-4.tif


  Sentinel shape: (370, 212, 752)
  Sentinel CRS: EPSG:4326
  Sentinel bounds: BoundingBox(left=-121.65308971048843, bottom=38.40998525532567, right=-121.58553640112264, top=38.42902953934901)
  Resampled CDL shape: (212, 752)
  Valid cropland pixels with target crops: 33
  Extracting 33 samples...


  Tile 4: 100%|██████████| 33/33 [00:00<00:00, 26389.33it/s]

  Tile 4 extracted: 33 samples

✅ NORTH CALIFORNIA saved to: C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\processed\california\north
   X shape: (1538, 36, 10)
   y shape: (1538,)


 ### 6. COMBINE SOUTH + NORTH FOR FULL CALIFORNIA

In [8]:
print("\n" + "="*60)
print("STEP 6: COMBINING SOUTH + NORTH CALIFORNIA")
print("="*60)

all_X = []
all_masks = []
all_y = []

if X_south is not None:
    all_X.append(X_south)
    all_masks.append(masks_south)
    all_y.append(y_south)
    print(f"  South: {X_south.shape[0]} samples")

if X_north is not None:
    all_X.append(X_north)
    all_masks.append(masks_north)
    all_y.append(y_north)
    print(f"  North: {X_north.shape[0]} samples")

if len(all_X) > 0:
    X_combined = np.concatenate(all_X, axis=0)
    masks_combined = np.concatenate(all_masks, axis=0)
    y_combined = np.concatenate(all_y, axis=0)
    
    print(f"\nCombined California dataset:")
    print(f"  X shape: {X_combined.shape}")
    print(f"  masks shape: {masks_combined.shape}")
    print(f"  y shape: {y_combined.shape}")
    print(f"  Total samples: {X_combined.shape[0]:,}")
    
    # Save combined data
    np.save(os.path.join(OUTPUT_DIR_COMBINED, 'X.npy'), X_combined)
    np.save(os.path.join(OUTPUT_DIR_COMBINED, 'masks.npy'), masks_combined)
    np.save(os.path.join(OUTPUT_DIR_COMBINED, 'y.npy'), y_combined)
    
    print(f"\n✅ Combined data saved to: {OUTPUT_DIR_COMBINED}")
    
    # Class distribution
    unique, counts = np.unique(y_combined, return_counts=True)
    print("\nClass distribution (combined):")
    for code, count in zip(unique, counts):
        print(f"  Crop {code}: {count:,} samples ({100*count/len(y_combined):.1f}%)")
else:
    print("⚠️ No data extracted for any region!")


STEP 6: COMBINING SOUTH + NORTH CALIFORNIA
  South: 5000 samples
  North: 1538 samples

Combined California dataset:
  X shape: (6538, 36, 10)
  masks shape: (6538, 36, 10)
  y shape: (6538,)
  Total samples: 6,538

✅ Combined data saved to: C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\processed\california

Class distribution (combined):
  Crop 36: 298 samples (4.6%)
  Crop 57: 98 samples (1.5%)
  Crop 69: 4,280 samples (65.5%)
  Crop 74: 1 samples (0.0%)
  Crop 204: 1,861 samples (28.5%)


# 7. FINAL SUMMARY

In [9]:
print("\n" + "="*60)
print("DATA EXTRACTION COMPLETE!")
print("="*60)
print(f"""
Summary:
  - South California: {X_south.shape[0] if X_south is not None else 0} samples
  - North California: {X_north.shape[0] if X_north is not None else 0} samples
  - Total California: {X_combined.shape[0] if len(all_X) > 0 else 0} samples
  - Each sample: 36 time steps × 10 spectral bands

Files saved:
  - South: {OUTPUT_DIR_SOUTH}
  - North: {OUTPUT_DIR_NORTH}
  - Combined: {OUTPUT_DIR_COMBINED}

Next step:
  Run notebook: 02_eda_visuals.ipynb
  to explore the data and generate visualizations (like paper Figure 2)
""")


DATA EXTRACTION COMPLETE!

Summary:
  - South California: 5000 samples
  - North California: 1538 samples
  - Total California: 6538 samples
  - Each sample: 36 time steps × 10 spectral bands

Files saved:
  - South: C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\processed\california\south
  - North: C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\processed\california\north
  - Combined: C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\processed\california

Next step:
  Run notebook: 02_eda_visuals.ipynb
  to explore the data and generate visualizations (like paper Figure 2)

